# LoRA 가중치 저장 & HuggingFace 업로드

**목적:** Fine-tuning된 LoRA 가중치를 저장하고, HuggingFace Hub에 업로드하여 누구나 사용할 수 있게 공유합니다.

**흐름:**
1. 패키지 설치 & HuggingFace 로그인
2. 베이스 모델 로드 & LoRA 학습 (이전 실습 복습)
3. LoRA 가중치 저장
4. HuggingFace Hub에 업로드
5. 업로드된 모델 다운로드 & 테스트

> ⚠️ 런타임 → 런타임 유형 변경 → **T4 GPU** 선택 후 실행하세요.

## 1. 패키지 설치

In [ ]:
!pip install -q --upgrade datasets
!pip install -q unsloth
!pip install -q huggingface_hub

## 2. HuggingFace 로그인

HuggingFace에 모델을 업로드하려면 **Access Token**이 필요합니다.

1. [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) 에서 토큰 생성
2. 권한: **Write** 권한 필요
3. 아래 셀 실행 후 토큰 입력

In [ ]:
from huggingface_hub import login
login()

## 3. 모델 로드 & LoRA 학습

이전 실습(`nutriday_eval_practice.ipynb`)과 동일한 과정입니다.  
이미 학습된 가중치가 있다면 **섹션 4로 건너뛰세요.**

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)
print("✅ 모델 & LoRA 설정 완료")

In [ ]:
from google.colab import files
uploaded = files.upload()  # nutriday_train.json 선택

In [ ]:
import json
from datasets import Dataset

with open("nutriday_train.json", "r", encoding="utf-8") as f:
    train_raw = json.load(f)

def format_example(example):
    messages = [
        {"role": "user",      "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

dataset = Dataset.from_list(train_raw).map(format_example)
print(f"✅ 학습 데이터셋 준비 완료: {len(dataset)}개")

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = 1024,
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        fp16 = True,
        logging_steps = 10,
        output_dir = "./nutriday_output",
        report_to = "none",
    ),
)

trainer.train()
print("✅ 학습 완료")

## 4. LoRA 가중치 저장

학습된 LoRA 가중치만 따로 저장합니다. 베이스 모델(약 3GB)은 저장하지 않고, **변화량(LoRA 어댑터)만 저장**하므로 용량이 매우 작습니다 (수십 MB).

### 저장되는 파일들

| 파일 | 역할 |
|------|------|
| `adapter_config.json` | LoRA 설정 (r, alpha, target_modules 등) |
| `adapter_model.safetensors` | 학습된 LoRA 가중치 |
| `tokenizer.json`, `tokenizer_config.json` 등 | 토크나이저 파일들 |

In [ ]:
SAVE_DIR = "nutriday_lora"

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"✅ LoRA 가중치 저장 완료 → {SAVE_DIR}/")

import os
print("\n--- 저장된 파일 목록 ---")
for f in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(os.path.join(SAVE_DIR, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

## 5. HuggingFace Hub에 업로드

### 업로드 방식 비교

| 방식 | 설명 | 언제 사용? |
|------|------|-----------|
| `model.push_to_hub()` | 모델 객체에서 직접 업로드 | 학습 직후, 모델이 메모리에 있을 때 |
| `HfApi.upload_folder()` | 저장된 폴더를 통째로 업로드 | 저장된 파일을 나중에 업로드할 때 |

아래에서는 두 가지 방법을 모두 보여줍니다.

### 방법 A: `push_to_hub()` — 모델이 메모리에 있을 때 (권장)

`REPO_ID`를 본인의 HuggingFace 사용자명으로 변경하세요.  
예: `"my-username/nutriday-qwen2.5-1.5b-lora"`

In [ ]:
REPO_ID = "my-username/nutriday-qwen2.5-1.5b-lora"  # ← 본인 사용자명으로 변경!

model.push_to_hub(REPO_ID, private=True)
tokenizer.push_to_hub(REPO_ID, private=True)

print(f"✅ 업로드 완료! → https://huggingface.co/{REPO_ID}")

### 방법 B: `upload_folder()` — 저장된 폴더에서 업로드

모델이 메모리에 없고, 이미 저장된 LoRA 폴더만 있을 때 사용합니다.

In [ ]:
# from huggingface_hub import HfApi
#
# api = HfApi()
# api.create_repo(REPO_ID, private=True, exist_ok=True)
# api.upload_folder(
#     folder_path = "nutriday_lora",
#     repo_id = REPO_ID,
# )
# print(f"✅ 폴더 업로드 완료! → https://huggingface.co/{REPO_ID}")

## 6. 업로드된 모델 다운로드 & 테스트

다른 환경에서 업로드된 LoRA 가중치를 불러와 사용하는 방법입니다.  
실제로 HuggingFace에서 다운로드받아 동작하는지 확인합니다.

In [ ]:
from unsloth import FastLanguageModel
from peft import PeftModel

# 1) 베이스 모델 로드
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

# 2) HuggingFace에서 LoRA 가중치 다운로드 & 적용
model = PeftModel.from_pretrained(base_model, REPO_ID)

print("✅ HuggingFace에서 모델 로드 완료!")

In [ ]:
def generate(prompt, model, max_new_tokens=200):
    FastLanguageModel.for_inference(model)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.7,
        do_sample=True,
    )
    return tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

test_questions = [
    "뉴트리데이 유산균 어떤 제품이 있나요?",
    "배송은 얼마나 걸리나요?",
    "환불하고 싶어요.",
]

for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {generate(q, model)}")
    print("-" * 50)

print("\n✅ HuggingFace에서 다운로드한 모델이 정상 동작합니다!")

## 정리

### 오늘 배운 것

| 단계 | 함수 | 설명 |
|------|------|------|
| 저장 | `model.save_pretrained()` | LoRA 가중치를 로컬에 저장 |
| 업로드 | `model.push_to_hub()` | 메모리의 모델을 HuggingFace에 직접 업로드 |
| 업로드 | `HfApi.upload_folder()` | 저장된 폴더를 HuggingFace에 업로드 |
| 다운로드 | `PeftModel.from_pretrained()` | HuggingFace에서 LoRA 가중치를 다운로드하여 적용 |

### LoRA vs 전체 모델 저장

| | LoRA 어댑터만 | 전체 모델 (merged) |
|---|---|---|
| 용량 | ~20-50 MB | ~3 GB (1.5B 모델 기준) |
| 저장 방법 | `save_pretrained()` | `model.save_pretrained_merged()` |
| 불러오기 | 베이스 모델 + `PeftModel.from_pretrained()` | `from_pretrained()`만으로 가능 |
| 장점 | 가볍고, 베이스 모델 공유 가능 | 독립적, 추가 의존성 없음 |
| 단점 | 베이스 모델을 따로 받아야 함 | 용량이 큼 |